# Kaggle training
Select one configuration or bounded tuning job. Defaults only generate a dry-run manifest. All implementations are imported by repository scripts.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
INPUT_DIR = Path(os.environ.get("SEQREC_INPUT_DIR", "/kaggle/input"))
WORK_DIR = Path(os.environ.get("SEQREC_WORK_DIR", "/kaggle/working"))
REPO = Path(os.environ.get("SEQREC_REPO_DIR", str(WORK_DIR / "sequential-recommendation")))
PYTHON = os.environ.get("SEQREC_PYTHON", str(WORK_DIR / "seqrec-env/bin/python"))
if not Path(PYTHON).exists():
    PYTHON = sys.executable
def run(*args):
    subprocess.run([PYTHON, *map(str, args)], cwd=REPO, check=True)
print({"repo": str(REPO), "python": PYTHON, "input": str(INPUT_DIR), "work": str(WORK_DIR)})


In [ ]:
SESSION_FILE = WORK_DIR / "seqrec_session.json"
session = json.loads(SESSION_FILE.read_text())
REPO, PYTHON, BUNDLE = Path(session["repo"]), session["python"], Path(session["bundle"])
PROFILE = "core_research"  # pilot | core_research | full_research
MODE = "dry-run"  # dry-run | tune | train | analysis | benchmark
EXPERIMENT = "E1"
DATASET = session["dataset"]
MODEL = "sasrec_ce"
VARIANT = None  # A1-A4 for E2; F0-F5 for E4
SEED = 7
RETENTION = None
MAX_SEQ_LENGTH = None
DEVICE = "cuda:0"
FINAL_SEEDS = False
SELECTION = None  # path to selected_config.yaml from validation-only tuning
RESUME = True
RESULTS_ROOT = REPO / ("results/pilots_phase5a" if PROFILE == "pilot" else "results/core" if PROFILE == "core_research" else "results/runs")
OVERRIDES = []  # e.g. ['batch_size=64','gradient_accumulation_steps=2'] for independent CE/BCE only


In [ ]:
run("scripts/check_environment.py")
run("scripts/verify_data.py", "--bundle", BUNDLE)
if MODE == "benchmark":
    run("scripts/benchmark_pipeline.py", "--bundle", BUNDLE, "--device", DEVICE,
        "--output", "results/phase5a/kaggle_benchmark_session1", "--batches", "20", "--repeats", "2", "--full-validation")
elif MODE == "tune":
    command = ["scripts/run_tuning.py", "--profile", PROFILE, "--bundle", BUNDLE, "--model", MODEL,
               "--device", DEVICE, "--results-root", "results/tuning"]
    if RESUME: command.append("--resume")
    run(*command)
else:
    command = ["scripts/run_experiment.py", "--profile", PROFILE, "--experiment", EXPERIMENT,
               "--dataset", DATASET, "--bundle", BUNDLE, "--device", DEVICE, "--results-root", RESULTS_ROOT]
    if MODE == "dry-run": command += ["--dry-run", "--output", "results/tables/kaggle_manifest.csv"]
    if MODE not in ("dry-run", "train", "analysis"): raise ValueError("Unknown MODE")
    if MODE != "analysis":
        if MODEL: command += ["--model", MODEL]
        if VARIANT: command += ["--variant", VARIANT]
        if SEED is not None: command += ["--seed", str(SEED)]
    if FINAL_SEEDS: command.append("--final-seeds")
    if RETENTION is not None: command += ["--retention", str(RETENTION)]
    if MAX_SEQ_LENGTH is not None: command += ["--max-seq-length", str(MAX_SEQ_LENGTH)]
    if SELECTION: command += ["--selection", str(SELECTION)]
    if RESUME: command.append("--resume")
    for override in OVERRIDES: command += ["--set", override]
    run(*command)


In [ ]:
# Review utilization while working (training CLIs write epoch memory and timings).
subprocess.run(["nvidia-smi"], check=False)
subprocess.run([PYTHON, "scripts/verify_results.py", "--root", str(RESULTS_ROOT)], cwd=REPO, check=False)
# Export explicitly before the session expires; choose a new name each time.
EXPORT_PATH = WORK_DIR / "seqrec-results-session1.zip"
run("scripts/export_results.py", "--root", REPO / "results", "--output", EXPORT_PATH)
print("Download or Save Version with output:", EXPORT_PATH)
